<a name="pyclasses"></a>
## Classes

Objects are an encapsulation of variables and functions into a single entity. Objects get their variables and functions from classes. Classes are essentially a template to create your objects.

This is an example for a basic class named ``Car`` with four class variables and a class function (= method). The last line instantiates a variable ``mcqueen`` that holds an object of type ``Car``:

In [ ]:
class Car:
    name = ""
    kind = ""
    color = ""
    value = 100000.00

    def description(self):
        description_str = f"{self.name} is a {self.color} {self.kind} worth {self.value:.2f} €."
        return description_str


mcqueen = Car()

Set the values of variable ``mcqueen`` and call the object's function:

In [ ]:
mcqueen.name = "Lightning McQueen"
mcqueen.color = "red"
mcqueen.kind = "Corvette"

mcqueen_descr = mcqueen.description()
print(mcqueen_descr)

### `__init__()`

The example above is a class and object in their simplest form as they are rarely used in real life applications. All Python classes have a built-in method called ``__init__()``, which is always executed automatically when an object of the class is created (instantiated). The ``__init__()`` method is used to assign values to object properties, or other operations that are necessary to do when the object is being created:

In [ ]:
class Car:

    def __init__(self, name, kind, color):
        self.name = name
        self.kind = kind
        self.color = color
        self.value = 100000.00

    def description(self):
        description_str = f"{self.name} is a {self.color} {self.kind} worth {self.value:.2f} €."
        return description_str


mcqueen = Car('Lightning McQueen', 'Corvette', 'red')
print(f"{mcqueen.description()=}")
print(f"{mcqueen.name=}")

### The self Parameter
The ``self`` parameter is a reference to the current instance of the class. It is used to access variables that belong to the class.

It does not have to be named ``self``, but it has to be the first parameter of any function in the class.

### Other special method names 

Apart from `__init__`, there are other reserved special names as well, like `__del__` and `__doc__`. See the [Python Docs](https://docs.python.org/3/reference/datamodel.html#basic-customization) for more information.

> Note: In python you'll meet *dunder* methods quite frequently. These are methods that start and end with a double underscore like `__init__`. Dunder methods also specify how an object behaves under operations like `+`, `-`, `==` but also how builtin functions like `len` work on an object. Otherwise, integers could not simply be added or the `+` operator could not imply totally different behaviors for floats and lists.

### Access modifiers

Access modifiers (like `private` or `public`) are not implemented as such in python classes. A hint is normally used, where `private` members are named with a leading underscore `_`.

In [ ]:
class PrivateCar:
    kind = ""
    color = ""
    _owner = "Johnathan"  # owner shall be private



privateMcQueen = PrivateCar()
privateMcQueen._owner = "Jack"  # normally would expect a violation here

### Exercise:
One can, indeed create a workaround to make attributes private by overriding the class's dunder methods for getting and setting attributes: `__setattr__` and `__getattribute__`. 

Give that a try and make `_owner` truly private.

### Solution Remarks:
It turns out that it seems rather impossible to make a class attribute properly secret. Even when overriding get and set methods, one is still able to obtain the private attribute by `object.__getattribute__(obj, attribute_name)`. 

See here:

In [ ]:
### BEGIN SOLUTION
# STUB: class ReallyPrivateCar:
# STUB:     ...
class ReallyPrivateCar:

    def __init__(self, kind, color, owner):
        self.kind = kind
        self.color = color
        object.__setattr__(self, "_owner", owner)

    def __getattribute__(self, attr):
        if attr == "_owner":
            raise AttributeError("_owner is private")
        else:
            return super().__getattribute__(attr)

    def __setattr__(self, attr, value):
        if attr == "_owner":
            raise AttributeError("_owner is private")
        else:
            super().__setattr__(attr, value)



really_private_McQueen = ReallyPrivateCar("porsche", "red", "Jack")
try:
    really_private_McQueen._owner = "Jack"  # expect AttributeError
except AttributeError as e:
    print(e)
    print("But we can still get it:")
    print(f"{object.__getattribute__(really_private_McQueen, '_owner')=}")
### END SOLUTION

However, someone calling `object.__getattribute__` to handle an object attribute occurs rarely. Assuming the setting, that we want to protect the private attribute from being meddled with carelessly or from being overridden we can take the risk of `object.__getattribute__` working anyway. **Note, that `object.__getattribute__` will even work when we inspect the call stack and would try to guard outside class scope calls.**

In case of true privacy/security matters, other measures have to be taken, as e.g. encrypting values and not saving the attribute on the class instance directly. 

Anyway, to make the previous solution a bit easier to work with from inside the class, we could write a thin wrapper around `object.__getattribute__`.

In [ ]:
class ReallyPrivateCar:

    def __init__(self, kind, color, owner):
        self.kind = kind
        self.color = color
        self.__set_owner(owner)
        # object.__setattr__(self, "_owner", owner)

    def __get_owner(self):
        """To make life easier inside the class."""
        return object.__getattribute__(self, "_owner")

    def __set_owner(self, owner):
        """To make life easier inside the class."""
        object.__setattr__(self, "_owner", owner)

    def __sell_car(self, old, new):
        """Assume this would be an appropriate in class case"""
        if not self.__get_owner() == old:
            raise ValueError("Cannot sell car, as the specified seller is not its owner")
        self.__set_owner(new)


    def __getattribute__(self, attr):
        if attr == "_owner":
            raise AttributeError("_owner is private")
        else:
            return super().__getattribute__(attr)

    def __setattr__(self, attr, value):
        if attr == "_owner":
            raise AttributeError("_owner is private")
        else:
            super().__setattr__(attr, value)



really_private_McQueen = ReallyPrivateCar("porsche", "red", "Jack")
try:
    really_private_McQueen._owner = "Jack"  # expect AttributeError
except AttributeError as e:
    print(e)
    print("But we can still get it:")
    print(f"{object.__getattribute__(really_private_McQueen, '_owner')=}")

### Class and Instance Variables
Source: [Python Docs](https://docs.python.org/3/tutorial/classes.html#class-and-instance-variables).

There are two types of variables within a Python Classes, that behave differently:
- Class Variables
- Instance Variables

In [ ]:
class Dog:

    kind = 'canine'  # class variable shared by all instances

    def __init__(self, name):
        self.name = name  # instance variable unique to each instance

In [ ]:
d = Dog('Fido')
e = Dog('Buddy')

In [ ]:
d.kind

In [ ]:
e.kind

In [ ]:
d.name

In [ ]:
e.name

__Attention:__ Don't mix Class and Instance variables up!

In [ ]:
class TrickDog:

    tricks = []  # mistaken use of a class variable

    def __init__(self, name):
        self.name = name

    def add_trick(self, trick):
        self.tricks.append(trick)

In [ ]:
d = TrickDog('Fido')
e = TrickDog('Buddy')
d.add_trick('roll over')
e.add_trick('play dead')

In [ ]:
e.tricks

__How to make it right__

In [ ]:
class WinningTrickDog:

    def __init__(self, name):
        self.name = name
        self.tricks = []  # creates a new empty list for each dog

    def add_trick(self, trick):
        self.tricks.append(trick)

In [ ]:
d = WinningTrickDog('Fido')
e = WinningTrickDog('Buddy')
d.add_trick('roll over')
e.add_trick('play dead')

In [ ]:
d.tricks

In [ ]:
e.tricks

>Note: There is also a difference on instance and class *methods*.

The previous can serve as a good example to shortly speak about class *inheritance*, the basic syntax being 

```python
class DerivedClassName(BaseClassName):
    ...
```

### Exercise: Try to rewrite the `WinningTrickDog` (or `TrickDog`) class to inherit from the `Dog` class

In [ ]:
### BEGIN SOLUTION
# STUB: class WinningTrickDog(Dog):
# STUB:     ...
class WinningTrickDog(Dog):

    def __init__(self, name):
        super().__init__(name)
        self.tricks = []

    def add_trick(self, trick):
        self.tricks.append(trick)

bobby = WinningTrickDog("Bobby")
print(bobby.name)
print(bobby.tricks)
bobby.add_trick("sit")
print(bobby.tricks)
### END SOLUTION

### Short Note on an Object's Lifetime

Objects are never explicitly destroyed; however, when they become unreachable they may be garbage-collected. An implementation is allowed to postpone garbage collection or omit it altogether — it is a matter of implementation quality how garbage collection is implemented, as long as no objects are collected that are still reachable.

__Source:__ https://docs.python.org/3.13/reference/datamodel.html#objects-values-and-types